# 게임 유저 이탈 분석 리포트

- **분석가:** 황동현  
- **데이터:** Online Gaming Behavior Dataset (40,034명)  
- **분석 목적:** 이탈 유저 조기 식별 및 리텐션 액션 아이템 도출

---

## 1. 배경 및 문제 정의

게임 서비스에서 신규 유저 유입 비용은 기존 유저 유지 비용보다
평균 5배 이상 높다. 따라서 이탈 유저를 사전에 식별하고
리텐션 캠페인을 집행하는 것이 비용 효율적인 전략이다.

본 분석은 40,034명의 유저 행동 데이터를 바탕으로
**어떤 유저가 이탈할 가능성이 높은지**를 파악하고
**구체적인 액션 아이템**을 도출하는 것을 목표로 한다.

- 전체 이탈률: **25.8%** (10,324명 / 40,034명)
- 이탈 정의: EngagementLevel = 'Low' 인 유저

---

## 2. 핵심 발견 — 이탈을 결정하는 건 "접속 빈도"

7개 가설을 검증한 결과, 대부분의 가설이 기각되었고
**접속 빈도(SessionsPerWeek)** 가 유일하게 압도적인 영향을 미쳤다.

### 기각된 가설 (영향 없음)
| 변수 | 이탈률 범위 | 판정 |
|------|------------|------|
| 플레이 시간 | 25.2% ~ 26.2% | ❌ 기각 |
| 결제 여부 | 25.3% ~ 25.9% | ❌ 기각 |
| 게임 장르 | 25.0% ~ 26.4% | ❌ 기각 |
| 게임 난이도 | 25.3% ~ 26.0% | ❌ 기각 |

### 채택된 가설 (유의미한 영향)
| 변수 | 이탈률 범위 | 판정 |
|------|------------|------|
| 주간 세션 수 | 10.4% ~ 84.4% | ✅ 채택 |
| 레벨 구간 | 22.4% ~ 30.4% | ✅ 채택 |
| 세션 빈도+길이 조합 | 3.1% ~ 63.7% | ✅ 채택 |

---

## 3. 상세 분석 결과

### 3-1. 주간 세션 수 — 이탈의 핵심 변수

주 1~2회 접속 유저의 이탈률은 **84.4%** 로
주 7회 이상 접속 유저(10.4%)보다 **8배 높다.**

→ 접속 습관이 형성되지 않은 유저가 이탈 위험군의 핵심

### 3-2. 레벨 구간별 이탈 허들

레벨 1~10 구간 이탈률 **30.4%** 로 가장 높고
레벨이 올라갈수록 이탈률이 꾸준히 감소한다.

→ 초반 온보딩 경험이 장기 잔존률을 결정

### 3-3. 유저 유형별 이탈률

| 유저 유형 | 이탈률 |
|----------|--------|
| 고빈도·장시간 (헤비) | 3.1% |
| 고빈도·단시간 | 15.2% |
| 저빈도·장시간 | 28.1% |
| 저빈도·단시간 (라이트) | 63.7% |

→ 헤비 유저와 라이트 유저의 이탈률 차이가 **20배**

---

## 4. 모델링 결과 — 로지스틱 회귀

| 지표 | 값 |
|------|-----|
| 전체 정확도 | 88% |
| 이탈 유저 Recall | 68% |
| 이탈 유저 Precision | 81% |

### 변수 중요도 (p-value 검증)

| 순위 | 변수 | 계수 | p-value | 해석 |
|------|------|------|---------|------|
| 1 | SessionsPerWeek | -2.054 | 0.000 | 접속 빈도 높을수록 이탈 급감 |
| 2 | AvgSessionDurationMinutes | -1.378 | 0.000 | 세션 길수록 이탈 감소 |
| 3 | AchievementsUnlocked | -0.319 | 0.000 | 업적 많을수록 이탈 감소 |
| 4 | PlayerLevel | -0.300 | 0.000 | 레벨 높을수록 이탈 감소 |
| - | PlayTimeHours | +0.018 | 0.244 | 통계적으로 유의미하지 않음 |
| - | InGamePurchases | -0.013 | 0.417 | 통계적으로 유의미하지 않음 |
| - | Gender / Genre / Difficulty | ≈0 | >0.05 | 통계적으로 유의미하지 않음 |

EDA에서 도출한 인사이트가 모델에서도 동일하게 확인되었다.

---

## 5. 이탈 위험 세그먼트

| 위험 구간 | 비율 | 설명 |
|----------|------|------|
| 고위험 (60~100%) | 17.4% | 즉시 개입 필요 |
| 중위험 (30~60%) | 15.0% | 모니터링 필요 |
| 저위험 (0~30%) | 67.6% | 현상 유지 |

전체 유저의 **17.4%** 가 고위험군으로 분류되었다.

---

## 6. 액션 아이템

### 단기 (즉시 실행 가능)
1. **주간 세션 수 2회 이하 유저 자동 감지**
   → 접속 3일 경과 시 푸시 알림·복귀 보상 자동 발송
   → 예상 효과: 고위험군 17.4% 중 일부를 중위험으로 전환

2. **레벨 1~10 구간 온보딩 강화**
   → 초반 7일 내 보상 증가, 튜토리얼 간소화
   → 초반 이탈률 30.4% → 목표 25% 이하로 감소

### 중기 (1~3개월)
3. **유저 유형별 리텐션 전략 분리**
   → 라이트 유저(63.7%): 접속 유도 중심 캠페인
   → 헤비 유저(3.1%): 이탈 방지보다 과금 전환 집중

4. **헤비 유저 정의 기준 수립**
   → 주 7회 이상 + 세션 60분 이상 유저를 VIP로 분류
   → 별도 혜택 제공으로 장기 유지